<a href="https://colab.research.google.com/github/shivalakshmanan/ml-capstone26/blob/main/CapstoneR1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#generate_data.py

In [39]:
%%writefile part_a_clean_eda.py
import os
import pandas as pd
import numpy as np

os.makedirs("outputs", exist_ok=True)

# 1. Read input data
raw_df = pd.read_csv("data/raw_shipments.csv")
hubs_df = pd.read_csv("data/hubs.csv")

# 2. Extract "Before" metrics after removing duplicates but before any cleaning/filling
dup_free_raw = raw_df.drop_duplicates().copy()
missing_distance_before = dup_free_raw["distance_km"].isna().sum()

# missing_parcel_weight_before does not include sentinel rows (> 100)
weight_before_subset = dup_free_raw[~(dup_free_raw["parcel_weight_kg"] > 100)]
missing_parcel_weight_before = weight_before_subset["parcel_weight_kg"].isna().sum()

missing_declared_value_before = dup_free_raw["declared_value"].isna().sum()
missing_payment_mode_before = dup_free_raw["payment_mode"].isna().sum()

service_type_spellings_before = dup_free_raw["service_type"].nunique()

# Define internal implementations of shipment_prep helper functions to keep script self-contained
PROMISED_HOURS = {"Express": 36, "Standard": 60, "Economy": 96}
FEATURES = [
    "promised_hours", "distance_km", "parcel_weight_kg", "log_declared_value", "n_stops",
    "rain_flag", "sale_event_flag", "is_cod", "is_weekend", "dispatch_hour",
    "hub_tier", "hub_daily_capacity",
    "label_print_batch", "scanner_build_no", "tape_roll_no", "sticker_serial_mod", "shelf_slot_no",
]

def clean_shipments(raw, medians=None):
    clean_df = raw.drop_duplicates().copy()
    clean_df["service_type"] = clean_df["service_type"].str.strip().str.title()
    clean_df["payment_mode"] = clean_df["payment_mode"].fillna("Unknown")
    clean_df["declared_value_inr"] = clean_df["declared_value"].str.replace(r'[^\d.]', '', regex=True).astype(float)
    clean_df["dispatch_time"] = pd.to_datetime(clean_df["dispatch_time"])
    clean_df.loc[clean_df["parcel_weight_kg"] > 40, "parcel_weight_kg"] = np.nan

    if medians is None:
        medians = {
            "distance_km": clean_df["distance_km"].median(),
            "parcel_weight_kg": clean_df["parcel_weight_kg"].median(),
            "declared_value_inr": clean_df["declared_value_inr"].median()
        }

    clean_df["distance_km"] = clean_df["distance_km"].fillna(medians["distance_km"])
    clean_df["parcel_weight_kg"] = clean_df["parcel_weight_kg"].fillna(medians["parcel_weight_kg"])
    clean_df["declared_value_inr"] = clean_df["declared_value_inr"].fillna(medians["declared_value_inr"])
    return clean_df, medians

def build_features(clean, hubs):
    hubs_df = hubs.copy()
    clean_df = clean.copy()
    merged_df = clean_df.merge(hubs_df, on="hub_id", how="left")
    merged_df["promised_hours"] = merged_df["service_type"].map(PROMISED_HOURS).astype(int)
    merged_df["is_cod"] = (merged_df["payment_mode"] == "COD").astype(int)
    merged_df["is_weekend"] = (merged_df["dispatch_time"].dt.dayofweek >= 5).astype(int)
    merged_df["dispatch_hour"] = merged_df["dispatch_time"].dt.hour.astype(int)
    merged_df["log_declared_value"] = np.log(merged_df["declared_value_inr"])
    return merged_df

# 3. Apply cleaning and feature engineering
clean_df, medians = clean_shipments(raw_df)
service_type_spellings_after = clean_df["service_type"].nunique()

modelling_df = build_features(clean_df, hubs_df)
modelling_df["is_delayed"] = (modelling_df["delivery_hours"] > modelling_df["promised_hours"]).astype(int)

# Save the modelling table
modelling_df.to_csv("data/modelling_table.csv", index=False)

# 4. Extract "After" metrics on modelling_df
target_cols = FEATURES + ["delivery_hours", "is_delayed"]
nulls_in_features_and_target = modelling_df[target_cols].isna().sum().sum()

declared_value_mean_inr = modelling_df["declared_value_inr"].mean()
declared_value_median_inr = modelling_df["declared_value_inr"].median()
log_declared_value_mean = modelling_df["log_declared_value"].mean()
log_declared_value_median = modelling_df["log_declared_value"].median()

# Calculate outliers boundaries on delivery_hours using IQR method
q25 = modelling_df["delivery_hours"].quantile(0.25)
q75 = modelling_df["delivery_hours"].quantile(0.75)
iqr = q75 - q25
hours_lower_bound = q25 - 1.5 * iqr
hours_upper_bound = q75 + 1.5 * iqr

# Compile checks into DataFrame and round values
checks_data = {
    "Metric": [
        "missing_distance_before",
        "missing_parcel_weight_before",
        "missing_declared_value_before",
        "missing_payment_mode_before",
        "service_type_spellings_before",
        "service_type_spellings_after",
        "nulls_in_features_and_target",
        "declared_value_mean_inr",
        "declared_value_median_inr",
        "log_declared_value_mean",
        "log_declared_value_median",
        "hours_lower_bound",
        "hours_upper_bound"
    ],
    "value": [
        round(float(missing_distance_before), 3),
        round(float(missing_parcel_weight_before), 3),
        round(float(missing_declared_value_before), 3),
        round(float(missing_payment_mode_before), 3),
        round(float(service_type_spellings_before), 3),
        round(float(service_type_spellings_after), 3),
        round(float(nulls_in_features_and_target), 3),
        round(float(declared_value_mean_inr), 3),
        round(float(declared_value_median_inr), 3),
        round(float(log_declared_value_mean), 3),
        round(float(log_declared_value_median), 3),
        round(float(hours_lower_bound), 3),
        round(float(hours_upper_bound), 3)
    ]
}

part_a_checks = pd.DataFrame(checks_data)
part_a_checks.to_csv("outputs/part_a_checks.csv", index=False)
print("Saved outputs/part_a_checks.csv")


Overwriting part_a_clean_eda.py


In [ ]:
"""Synthetic shipment data for the Delay Intelligence capstone.
Run once:  python generate_data.py   (writes data/hubs.csv, data/raw_shipments.csv, data/new_shipments.csv)
Do not edit any constant: every number in the brief was measured on this exact output."""
import os
import numpy as np
import pandas as pd

HUBS = [
    ("HUB01", "Mumbai", "West", 1, 1500), ("HUB02", "Delhi", "North", 1, 1600),
    ("HUB03", "Bengaluru", "South", 1, 1450), ("HUB04", "Hyderabad", "South", 1, 1350),
    ("HUB05", "Chennai", "South", 1, 1300), ("HUB06", "Pune", "West", 1, 1250),
    ("HUB07", "Kolkata", "East", 1, 1200), ("HUB08", "Ahmedabad", "West", 1, 1150),
    ("HUB09", "Jaipur", "North", 2, 950), ("HUB10", "Lucknow", "North", 2, 900),
    ("HUB11", "Chandigarh", "North", 2, 850), ("HUB12", "Indore", "West", 2, 820),
    ("HUB13", "Nagpur", "West", 2, 780), ("HUB14", "Coimbatore", "South", 2, 740),
    ("HUB15", "Kochi", "South", 2, 700), ("HUB16", "Surat", "West", 2, 650),
    ("HUB17", "Patna", "East", 3, 520), ("HUB18", "Ranchi", "East", 3, 480),
    ("HUB19", "Guwahati", "East", 3, 450), ("HUB20", "Raipur", "East", 3, 420),
    ("HUB21", "Bhubaneswar", "East", 3, 390), ("HUB22", "Dehradun", "North", 3, 350),
    ("HUB23", "Siliguri", "East", 3, 310), ("HUB24", "Agartala", "East", 3, 270),
]
DIST_MU = [180, 320, 480]      # mean lane distance (km) for hub tier 1 / 2 / 3
DIST_SD = [50, 90, 150]
STOPS_MU = [55, 70, 85]        # mean stops on the rider route for hub tier 1 / 2 / 3
HUB_EFFECT = [0, 2, 4]         # extra hours for hub tier 1 / 2 / 3
# How a hub really operates (1 = metro-like, 2 = regional, 3 = stretched). Four hubs operate unlike their tier label.
OPERATING_TIER = {"HUB07": 2, "HUB13": 3, "HUB15": 3, "HUB22": 2}
BASE_HOURS = {"Express": 10, "Standard": 28, "Economy": 58}


def simulate(n_per_hub, seed):
    """Clean shipments, n_per_hub rows for each of the 24 hubs, in hub order."""
    np.random.seed(seed)
    hubs = pd.DataFrame(HUBS, columns=["hub_id", "hub_city", "region", "hub_tier", "hub_daily_capacity"])
    n = n_per_hub * len(hubs)
    op_tier = [OPERATING_TIER.get(h, t) for h, t in zip(hubs["hub_id"], hubs["hub_tier"])]
    tier = np.repeat(op_tier, n_per_hub) - 1
    hub_id = np.repeat(hubs["hub_id"].values, n_per_hub)
    distance = np.clip(np.random.normal(np.array(DIST_MU)[tier], np.array(DIST_SD)[tier]), 15, 1500).round(1)
    stops = np.clip(np.random.normal(np.array(STOPS_MU)[tier], 10), 20, 140).round().astype(int)
    weight = np.clip(np.random.lognormal(0.6, 0.7, n), 0.1, 40).round(2)
    value = np.random.lognormal(7.0, 0.8, n).round().astype(int)
    service = np.random.choice(["Express", "Standard", "Economy"], n, p=[0.2, 0.55, 0.25])
    pay = np.random.choice(["Prepaid", "COD"], n, p=[0.6, 0.4])
    rain = (np.random.rand(n) < 0.18).astype(int)
    day = np.random.randint(0, 90, n)
    hour = np.random.randint(6, 22, n)
    minute = np.random.randint(0, 60, n)
    ts = pd.Timestamp("2026-07-01") + pd.to_timedelta(day, unit="D") + pd.to_timedelta(hour * 60 + minute, unit="m")
    sale = ((ts >= pd.Timestamp("2026-09-15")) & (ts < pd.Timestamp("2026-09-29"))).astype(int)
    weekend = (ts.dayofweek >= 5).astype(int)
    hours = (np.array([BASE_HOURS[s] for s in service]) + 0.03 * distance + 0.10 * stops
             + np.array(HUB_EFFECT)[tier] + 0.12 * weight + 4 * rain + 7 * sale
             + 3 * (pay == "COD") + 2 * weekend + 3 * (hour >= 18)
             + 12 * sale * (service == "Express") + 8 * (distance > 400) * rain
             + np.random.normal(0, 4, n)).round(2)
    return pd.DataFrame({
        "hub_id": hub_id, "service_type": service, "payment_mode": pay,
        "parcel_weight_kg": weight, "distance_km": distance, "declared_value": value,
        "n_stops": stops, "rain_flag": rain, "sale_event_flag": sale, "dispatch_time": ts,
        "label_print_batch": np.random.randint(1000, 9999, n),
        "scanner_build_no": np.random.randint(1, 40, n),
        "tape_roll_no": np.random.randint(1, 500, n),
        "sticker_serial_mod": np.random.uniform(0, 1, n).round(3),
        "shelf_slot_no": np.random.randint(1, 300, n),
        "delivery_hours": hours,
    })


def make_messy(df, seed, n_missing, n_case, n_sentinel, n_dupes, id_start):
    """Turn a clean table into a raw export: text columns, gaps, duplicates, a sentinel."""
    np.random.seed(seed)
    df = df.sample(frac=1.0, random_state=seed).reset_index(drop=True)
    n = len(df)
    df.insert(0, "shipment_id", ["SHP" + str(id_start + i) for i in range(n)])
    df["dispatch_time"] = df["dispatch_time"].dt.strftime("%Y-%m-%d %H:%M")
    df["declared_value"] = df["declared_value"].map(lambda v: "\u20b9" + format(v, ","))
    for col, k in [("distance_km", n_missing[0]), ("parcel_weight_kg", n_missing[1]),
                   ("declared_value", n_missing[2]), ("payment_mode", n_missing[3])]:
        df.loc[np.random.choice(n, k, replace=False), col] = np.nan
    pick = np.random.choice(n, n_case, replace=False)
    shapes = np.random.randint(0, 3, n_case)
    for i, s in zip(pick, shapes):
        v = df.at[i, "service_type"]
        df.at[i, "service_type"] = [v.upper(), v.lower(), "  " + v + " "][s]
    if n_sentinel:
        df.loc[np.random.choice(n, n_sentinel, replace=False), "parcel_weight_kg"] = 999.0
    if n_dupes:
        df = pd.concat([df, df.iloc[np.random.choice(n, n_dupes, replace=False)]], ignore_index=True)
    return df


if __name__ == "__main__":
    os.makedirs("data", exist_ok=True)
    hubs = pd.DataFrame(HUBS, columns=["hub_id", "hub_city", "region", "hub_tier", "hub_daily_capacity"])
    hubs.to_csv("data/hubs.csv", index=False)
    raw = make_messy(simulate(50, 2604), seed=7, n_missing=(36, 24, 12, 18),
                     n_case=120, n_sentinel=5, n_dupes=24, id_start=10001)
    raw.to_csv("data/raw_shipments.csv", index=False)
    new = make_messy(simulate(2, 99), seed=8, n_missing=(3, 2, 1, 2),
                     n_case=6, n_sentinel=0, n_dupes=0, id_start=90001)
    new = new.drop(columns=["delivery_hours"])
    new.to_csv("data/new_shipments.csv", index=False)
    print(len(hubs), len(raw), len(new))


24 1224 48


#Data Test

In [ ]:
import pandas as pd
import numpy as np

hubs_df = pd.read_csv("data/hubs.csv")
raw_df = pd.read_csv("data/raw_shipments.csv")
new_df = pd.read_csv("data/new_shipments.csv")

# hubs_df.head()
# print(hubs_df.isna().sum())
# raw_df.head()
# print(raw_df.isna().sum())
# new_df.head()
# print(new_df.isna().sum())

#PART A

#src/shipment_prep.py

In [ ]:
import numpy as np
import pandas as pd

# CONSTANTS
PROMISED_HOURS = {"Express": 36, "Standard": 60, "Economy": 96}
FEATURES = [
    "promised_hours", "distance_km", "parcel_weight_kg", "log_declared_value", "n_stops",
    "rain_flag", "sale_event_flag", "is_cod", "is_weekend", "dispatch_hour",
    "hub_tier", "hub_daily_capacity",
    "label_print_batch", "scanner_build_no", "tape_roll_no", "sticker_serial_mod", "shelf_slot_no",
]
NOISE_COLUMNS = ["label_print_batch", "scanner_build_no", "tape_roll_no", "sticker_serial_mod", "shelf_slot_no"]

def clean_shipments(raw, medians=None):
  clean_df = raw.drop_duplicates().copy()

  # clean dataset
  clean_df["service_type"] = clean_df["service_type"].str.strip().str.title()
  clean_df["payment_mode"] = clean_df["payment_mode"].fillna("Unknown")
  clean_df["declared_value_inr"] = clean_df["declared_value"].str.replace(r'[^\d.]', '', regex=True).astype(float)
  clean_df["dispatch_time"] = pd.to_datetime(clean_df["dispatch_time"])
  clean_df.loc[clean_df["parcel_weight_kg"] > 40, "parcel_weight_kg"] = np.nan

  # compute medians
  if medians is None:
    medians = {
        "distance_km": clean_df["distance_km"].median(),
        "parcel_weight_kg": clean_df["parcel_weight_kg"].median(),
        "declared_value_inr": clean_df["declared_value_inr"].median()
    }

  # fill values in dataset
  clean_df["distance_km"] = clean_df["distance_km"].fillna(medians["distance_km"])
  clean_df["parcel_weight_kg"] = clean_df["parcel_weight_kg"].fillna(medians["parcel_weight_kg"])
  clean_df["declared_value_inr"] = clean_df["declared_value_inr"].fillna(medians["declared_value_inr"])

  return clean_df, medians

  def build_features(clean, hubs):
  hubs_df = hubs.copy()
  clean_df = clean.copy()

  merged_df = clean_df.merge(hubs_df, on="hub_id", how="left")

  # Compute new columns on the merged shipment dataset
  merged_df["promised_hours"] = merged_df["service_type"].map(PROMISED_HOURS).astype(int)
  merged_df["is_cod"] = (merged_df["payment_mode"] == "COD").astype(int)
  merged_df["is_weekend"] = (merged_df["dispatch_time"].dt.dayofweek >= 5).astype(int)
  merged_df["dispatch_hour"] = merged_df["dispatch_time"].dt.hour.astype(int)
  merged_df["log_declared_value"] = np.log(merged_df["declared_value_inr"])

  return merged_df


#part_a_clean_eda.py

In [ ]:
import os
import pandas as pd
import numpy as np

os.makedirs("outputs", exist_ok=True)

# 1. Read input data
raw_df = pd.read_csv("data/raw_shipments.csv")
hubs_df = pd.read_csv("data/hubs.csv")

# 2. Extract "Before" metrics after removing duplicates but before any cleaning/filling
dup_free_raw = raw_df.drop_duplicates().copy()
missing_distance_before = dup_free_raw["distance_km"].isna().sum()

# missing_parcel_weight_before does not include sentinel rows (> 100)
weight_before_subset = dup_free_raw[~(dup_free_raw["parcel_weight_kg"] > 100)]
missing_parcel_weight_before = weight_before_subset["parcel_weight_kg"].isna().sum()

missing_declared_value_before = dup_free_raw["declared_value"].isna().sum()
missing_payment_mode_before = dup_free_raw["payment_mode"].isna().sum()

service_type_spellings_before = dup_free_raw["service_type"].nunique()

# 3. Apply cleaning and feature engineering
clean_df, medians = clean_shipments(raw_df)
service_type_spellings_after = clean_df["service_type"].nunique()

modelling_df = build_features(clean_df, hubs_df)
modelling_df["is_delayed"] = (modelling_df["delivery_hours"] > modelling_df["promised_hours"]).astype(int)

# Save the modelling table
modelling_df.to_csv("data/modelling_table.csv", index=False)

# 4. Extract "After" metrics on modelling_df
target_cols = FEATURES + ["delivery_hours", "is_delayed"]
nulls_in_features_and_target = modelling_df[target_cols].isna().sum().sum()

declared_value_mean_inr = modelling_df["declared_value_inr"].mean()
declared_value_median_inr = modelling_df["declared_value_inr"].median()
log_declared_value_mean = modelling_df["log_declared_value"].mean()
log_declared_value_median = modelling_df["log_declared_value"].median()

# Calculate outliers boundaries on delivery_hours using IQR method
q25 = modelling_df["delivery_hours"].quantile(0.25)
q75 = modelling_df["delivery_hours"].quantile(0.75)
iqr = q75 - q25
hours_lower_bound = q25 - 1.5 * iqr
hours_upper_bound = q75 + 1.5 * iqr

# Compile checks into DataFrame and round values
checks_data = {
    "Metric": [
        "missing_distance_before",
        "missing_parcel_weight_before",
        "missing_declared_value_before",
        "missing_payment_mode_before",
        "service_type_spellings_before",
        "service_type_spellings_after",
        "nulls_in_features_and_target",
        "declared_value_mean_inr",
        "declared_value_median_inr",
        "log_declared_value_mean",
        "log_declared_value_median",
        "hours_lower_bound",
        "hours_upper_bound"
    ],
    "value": [
        round(float(missing_distance_before), 3),
        round(float(missing_parcel_weight_before), 3),
        round(float(missing_declared_value_before), 3),
        round(float(missing_payment_mode_before), 3),
        round(float(service_type_spellings_before), 3),
        round(float(service_type_spellings_after), 3),
        round(float(nulls_in_features_and_target), 3),
        round(float(declared_value_mean_inr), 3),
        round(float(declared_value_median_inr), 3),
        round(float(log_declared_value_mean), 3),
        round(float(log_declared_value_median), 3),
        round(float(hours_lower_bound), 3),
        round(float(hours_upper_bound), 3)
    ]
}

part_a_checks = pd.DataFrame(checks_data)
part_a_checks.to_csv("outputs/part_a_checks.csv", index=False)
# display(part_a_checks)


#part_a_delay_by_group

In [ ]:
df = pd.read_csv("data/modelling_table.csv")

bins = [0, 200, 400, 600, 2000]
labels = ["0-200", "201-400", "401-600", "601+"]
df["distance_band"] = pd.cut(df["distance_km"], bins=bins, labels=labels, right=True)

# Define dimensions to analyze
dimensions = ["service_type", "hub_tier", "distance_band"]
all_groups = []

for dim in dimensions:
    # compute aggregates
    grouped = df.groupby(dim, observed=False).agg(
        shipments=("is_delayed", "count"),
        delayed=("is_delayed", "sum"),
        delay_rate=("is_delayed", "mean"),
        mean_hours=("delivery_hours", "mean")
    ).reset_index()

    # Standardize columns
    grouped.rename(columns={dim: "group"}, inplace=True)
    grouped.insert(0, "dimension", dim)
    all_groups.append(grouped)

# format coulmns
delay_by_group = pd.concat(all_groups, ignore_index=True)
delay_by_group["group"] = delay_by_group["group"].astype(str)
delay_by_group["shipments"] = delay_by_group["shipments"].astype(int)
delay_by_group["delayed"] = delay_by_group["delayed"].astype(int)
delay_by_group["delay_rate"] = delay_by_group["delay_rate"].round(4)
delay_by_group["mean_hours"] = delay_by_group["mean_hours"].round(4)


delay_by_group.to_csv("outputs/part_a_delay_by_group.csv", index=False)
# display(delay_by_group)


#part_a_correlations

In [ ]:
df_model = pd.read_csv("data/modelling_table.csv")

numeric_df = df_model.select_dtypes(include=["number"])

# Calculate correlation matrix
corr_matrix = numeric_df.corr().round(3)
corr_matrix_df = corr_matrix.reset_index().rename(columns={"index": "row lables"})

corr_matrix_df.to_csv("outputs/part_a_correlations.csv", index=False)
# display(corr_matrix_df)

#README - Data Findings

In [ ]:
# 1. Load the generated outputs
df_group = pd.read_csv("outputs/part_a_delay_by_group.csv")
df_corr = pd.read_csv("outputs/part_a_correlations.csv")
df_checks = pd.read_csv("outputs/part_a_checks.csv")

# --- Finding (i) ---
service_delays = df_group[df_group['dimension'] == 'service_type']
max_service_row = service_delays.loc[service_delays['delay_rate'].idxmax()]
finding_i = f"The service type delayed most often is **{max_service_row['group']}** with a delay rate of **{max_service_row['delay_rate']:.4f}**."

# --- Finding (ii) ---
hub_delays = df_group[df_group['dimension'] == 'hub_tier']
tier1_rate = hub_delays[hub_delays['group'] == '1']['delay_rate'].values[0]
tier2_rate = hub_delays[hub_delays['group'] == '2']['delay_rate'].values[0]
tier3_rate = hub_delays[hub_delays['group'] == '3']['delay_rate'].values[0]
finding_ii = f"The delay rate increases significantly as we move from Tier 1 to Tier 3 hubs: Tier 1 has a delay rate of **{tier1_rate:.4f}**, Tier 2 is **{tier2_rate:.4f}**, and Tier 3 is **{tier3_rate:.4f}**."

# --- Finding (iii) ---
dist_delays = df_group[df_group['dimension'] == 'distance_band']
dist_0_200 = dist_delays[dist_delays['group'] == '0-200']['delay_rate'].values[0]
dist_201_400 = dist_delays[dist_delays['group'] == '201-400']['delay_rate'].values[0]
dist_401_600 = dist_delays[dist_delays['group'] == '401-600']['delay_rate'].values[0]
dist_601 = dist_delays[dist_delays['group'] == '601+']['delay_rate'].values[0]
finding_iii = f"Across distance bands, the delay rate escalates linearly with distance: **{dist_0_200:.4f}** (0-200 km), **{dist_201_400:.3f}** (201-400 km), **{dist_401_600:.4f}** (401-600 km), and **{dist_601:.4f}** (601+ km)."

# --- Finding (iv) ---
# Row labels are stored in the first column of the correlations matrix. We renamed that column "row lables" or "feature" depending on execution. Let's find the correct label column name dynamically.
label_col = df_corr.columns[0]
df_corr_sorted = df_corr.copy()
df_corr_sorted['abs_corr'] = df_corr_sorted['delivery_hours'].abs()
# Exclude delivery_hours itself
df_corr_sorted = df_corr_sorted[df_corr_sorted[label_col] != 'delivery_hours']
top_3 = df_corr_sorted.sort_values(by='abs_corr', ascending=False).head(3)

ranked_corrs = []
for i, row in top_3.iterrows():
    ranked_corrs.append(f"{row[label_col]} ({row['delivery_hours']:.4f})")
finding_iv = f"The three columns most strongly correlated with `delivery_hours` are: " + ", ".join(ranked_corrs) + "."

# --- Finding (v) ---
# Get mean and median of declared_value_inr and log_declared_value from checks
val_mean = df_checks[df_checks['Metric'] == 'declared_value_mean_inr']['value'].values[0]
val_median = df_checks[df_checks['Metric'] == 'declared_value_median_inr']['value'].values[0]

log_mean = df_checks[df_checks['Metric'] == 'log_declared_value_mean']['value'].values[0]
log_median = df_checks[df_checks['Metric'] == 'log_declared_value_median']['value'].values[0]

finding_v = f"The `declared_value_inr` was log-transformed because the original column is highly right-skewed with a mean of **{val_mean:.4f}** and a median of **{val_median:.4f}**; after the transformation, the distribution is normalized with a mean of **{log_mean:.4f}** and median of **{log_median:.4f}**."

# Assemble README content
readme_content = f"""# README - Data Findings

### Core Findings from Part A Analysis

1. **Service Type Delays**
   {finding_i}

2. **Delay Rate by Hub Tier**
   {finding_ii}

3. **Delay Rate by Distance Band**
   {finding_iii}

4. **Top Correlations with Delivery Hours**
   {finding_iv}

5. **Declared Value Log Transformation**
   {finding_v}
"""

# Write README.md
with open("README.md", "w") as f:
    f.write(readme_content)

# print(readme_content)


Checked OK

#PART B

part_b_regression.py

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

# read data
data = pd.read_csv("data/modelling_table.csv")
X = data[FEATURES]
y = data["delivery_hours"]

# Call train_test_split
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

# Fit a StandardScaler and correctly transform the data sets
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# Calculate metrics
train_rows = float(len(X_train))
val_rows = float(len(X_val))
test_rows = float(len(X_test))

train_scaled_max_abs_mean = float(np.max(np.abs(np.mean(X_train_scaled, axis=0))))
test_scaled_max_abs_mean = float(np.max(np.abs(np.mean(X_test_scaled, axis=0))))

# Write outputs/part_b_checks.csv
checks_df = pd.DataFrame({
    "metric": [
        "train_rows",
        "val_rows",
        "test_rows",
        "train_scaled_max_abs_mean",
        "test_scaled_max_abs_mean"
    ],
    "value": [
        round(train_rows, 3),
        round(val_rows, 3),
        round(test_rows, 3),
        round(train_scaled_max_abs_mean, 3),
        round(test_scaled_max_abs_mean, 3)
    ]
})

import os
os.makedirs("outputs", exist_ok=True)
checks_df.to_csv("outputs/part_b_checks.csv", index=False)
# display(checks_df)

# Predict the mean of y_train for every validation row and record its metrics.
y_train_mean = np.mean(y_train)
y_val_mean_pred = np.full(len(y_val), y_train_mean)
val_mean_mse = mean_squared_error(y_val, y_val_mean_pred)
val_mean_rmse = np.sqrt(val_mean_mse)
val_mean_r2 = r2_score(y_val, y_val_mean_pred)

# print("Validation Mean MSE:", val_mean_mse)
# print("Validation Mean RMSE:", val_mean_rmse)
# print("Validation Mean R2:", val_mean_r2)



In [42]:
%%writefile part_b_regression.py
import os
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Ensure output directories exist
os.makedirs("outputs", exist_ok=True)

# Define the features exactly in the same pinned order
FEATURES = [
    "promised_hours", "distance_km", "parcel_weight_kg", "log_declared_value", "n_stops",
    "rain_flag", "sale_event_flag", "is_cod", "is_weekend", "dispatch_hour",
    "hub_tier", "hub_daily_capacity",
    "label_print_batch", "scanner_build_no", "tape_roll_no", "sticker_serial_mod", "shelf_slot_no",
]

# 1. Read input dataset
data = pd.read_csv("data/modelling_table.csv")
X = data[FEATURES]
y = data["delivery_hours"]

# 2. Split into Train, Validation, and Test (70/15/15 splits)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

# 3. Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# Extract and write regression split checks
train_rows = float(len(X_train))
val_rows = float(len(X_val))
test_rows = float(len(X_test))
train_scaled_max_abs_mean = float(np.max(np.abs(np.mean(X_train_scaled, axis=0))))
test_scaled_max_abs_mean = float(np.max(np.abs(np.mean(X_test_scaled, axis=0))))

checks_df = pd.DataFrame({
    "metric": [
        "train_rows",
        "val_rows",
        "test_rows",
        "train_scaled_max_abs_mean",
        "test_scaled_max_abs_mean"
    ],
    "value": [
        round(train_rows, 3),
        round(val_rows, 3),
        round(test_rows, 3),
        round(train_scaled_max_abs_mean, 3),
        round(test_scaled_max_abs_mean, 3)
    ]
})
checks_df.to_csv("outputs/part_b_checks.csv", index=False)

# 4. Decision Tree Depth Analysis
depths = [1, 2, 3, 4, 6, 8, 12, None]
depth_results = []
for d in depths:
    tree = DecisionTreeRegressor(max_depth=d, random_state=42)
    tree.fit(X_train_scaled, y_train)
    train_rmse = np.sqrt(mean_squared_error(y_train, tree.predict(X_train_scaled)))
    val_rmse = np.sqrt(mean_squared_error(y_val, tree.predict(X_val_scaled)))
    depth_str = "unlimited" if d is None else str(d)
    depth_results.append({
        "max_depth": depth_str,
        "train_rmse": round(train_rmse, 3),
        "val_rmse": round(val_rmse, 3)
    })

tree_depth_df = pd.DataFrame(depth_results)
tree_depth_df.to_csv("outputs/part_b_tree_depth.csv", index=False)

# Get optimal depth
otree_depth = int(tree_depth_df.loc[tree_depth_df['val_rmse'] == tree_depth_df['val_rmse'].min(), 'max_depth'].values[0])

# 5. Evaluate Candidate Models on Validation
models_metrics = []
def get_metrics(y_true, y_pred, model_name, split="val"):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    return {
        "model": model_name,
        "split": split,
        "MAE": round(mae, 3),
        "MSE": round(mse, 3),
        "RMSE": round(rmse, 3),
        "R2": round(r2, 3)
    }

# Model 1: Mean Baseline
y_train_mean = np.mean(y_train)
y_val_mean_pred = np.full(len(y_val), y_train_mean)
models_metrics.append(get_metrics(y_val, y_val_mean_pred, "mean_baseline"))

# Model 2: Linear Regression
lr = LinearRegression().fit(X_train_scaled, y_train)
models_metrics.append(get_metrics(y_val, lr.predict(X_val_scaled), "linear_regression"))

# Model 3: Decision Tree with optimal depth
dt = DecisionTreeRegressor(max_depth=otree_depth, random_state=42).fit(X_train_scaled, y_train)
models_metrics.append(get_metrics(y_val, dt.predict(X_val_scaled), "decision_tree"))

# Model 4: Random Forest
rf = RandomForestRegressor(n_estimators=100, random_state=42).fit(X_train_scaled, y_train)
models_metrics.append(get_metrics(y_val, rf.predict(X_val_scaled), "random_forest"))

# Model 5: Gradient Boosting (lr=0.1)
gb = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42).fit(X_train_scaled, y_train)
models_metrics.append(get_metrics(y_val, gb.predict(X_val_scaled), "gradient_boosting"))

# Save initial metrics
metrics_df = pd.DataFrame(models_metrics)
metrics_df.to_csv("outputs/part_b_metrics.csv", index=False)

# 6. Learning-rate Experiment
learn_rates = [0.01, 0.1, 1.0]
experiment_results = []
for lr_rate in learn_rates:
    gboost_exp = GradientBoostingRegressor(n_estimators=100, learning_rate=lr_rate, random_state=42).fit(X_train_scaled, y_train)
    preds = gboost_exp.predict(X_val_scaled)
    rmse = np.sqrt(mean_squared_error(y_val, preds))
    r2 = r2_score(y_val, preds)
    experiment_results.append({
        "learning_rate": lr_rate,
        "val_rmse": round(rmse, 3),
        "val_r2": round(r2, 3)
    })
pd.DataFrame(experiment_results).to_csv("outputs/part_b_gb_learning_rate.csv", index=False)

# 7. Cross-Validation on Train + Validation Set
X_train_val_scaled = np.concatenate([X_train_scaled, X_val_scaled], axis=0)
y_train_val = pd.concat([y_train, y_val], axis=0)
kf = KFold(n_splits=5, shuffle=True, random_state=42)

lr_model = LinearRegression()
lr_cv_scores = cross_val_score(lr_model, X_train_val_scaled, y_train_val, cv=kf, scoring="r2")
gb_model = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
gb_cv_scores = cross_val_score(gb_model, X_train_val_scaled, y_train_val, cv=kf, scoring="r2")

cv_results = pd.DataFrame([
    {
        "model": "linear_regression",
        "mean_r2": round(np.mean(lr_cv_scores), 3),
        "std_r2": round(np.std(lr_cv_scores), 3)
    },
    {
        "model": "gradient_boosting",
        "mean_r2": round(np.mean(gb_cv_scores), 3),
        "std_r2": round(np.std(gb_cv_scores), 3)
    }
])
cv_results.to_csv("outputs/part_b_cv.csv", index=False)

# 8. Score Final Model on Test Set
test_preds = lr.predict(X_test_scaled)
test_metrics = get_metrics(y_test, test_preds, "linear_regression", split="test")

metrics_df = pd.concat([metrics_df, pd.DataFrame([test_metrics])], ignore_index=True)
metrics_df.to_csv("outputs/part_b_metrics.csv", index=False)
print("Saved outputs/part_b_metrics.csv with final test score.")

Overwriting part_b_regression.py


#part_b_tree_depth.csv

In [ ]:
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error

depths = [1, 2, 3, 4, 6, 8, 12, None]
results = []

for d in depths:
    tree = DecisionTreeRegressor(max_depth=d, random_state=42)
    tree.fit(X_train_scaled, y_train)

    train_preds = tree.predict(X_train_scaled)
    val_preds = tree.predict(X_val_scaled)

    train_rmse = np.sqrt(mean_squared_error(y_train, train_preds))
    val_rmse = np.sqrt(mean_squared_error(y_val, val_preds))

    depth_str = "unlimited" if d is None else str(d)
    results.append({
        "max_depth": depth_str,
        "train_rmse": round(train_rmse, 3),
        "val_rmse": round(val_rmse, 3)
    })

tree_depth_df = pd.DataFrame(results)
tree_depth_df.to_csv("outputs/part_b_tree_depth.csv", index=False)
# display(tree_depth_df)

# optimal tree depth
otree_depth = int(tree_depth_df.loc[tree_depth_df['val_rmse'] == tree_depth_df['val_rmse'].min(), 'max_depth'].values[0])
# print(f"The calculated optimal tree depth is: {tree_depth}")


#Four candidate models & learning rate

In [ ]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Initialize lists to store metrics
models_metrics = []

# Helper to compute and return metric dictionary
def get_metrics(y_true, y_pred, model_name, split="val"):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    return {
        "model": model_name,
        "split": split,
        "MAE": round(mae, 3),
        "MSE": round(mse, 3),
        "RMSE": round(rmse, 3),
        "R2": round(r2, 3)
    }

# 1. Mean Baseline
baseline_metrics = get_metrics(y_val, y_val_mean_pred, "mean_baseline")
models_metrics.append(baseline_metrics)

# 2. Linear Regression Pipeline
reg_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LinearRegression())
]).fit(X_train, y_train)
reg_preds = reg_pipe.predict(X_val)
models_metrics.append(get_metrics(y_val, reg_preds, "linear_regression"))

# 3. Decision Tree (using optimal depth calculated in Task 4, which is 6)
dtree = DecisionTreeRegressor(max_depth=otree_depth, random_state=42).fit(X_train_scaled, y_train)
dtree_preds = dtree.predict(X_val_scaled)
models_metrics.append(get_metrics(y_val, dtree_preds, "decision_tree"))

# 4. Random Forest
rforest = RandomForestRegressor(n_estimators=100, random_state=42).fit(X_train_scaled, y_train)
rforest_preds = rforest.predict(X_val_scaled)
models_metrics.append(get_metrics(y_val, rforest_preds, "random_forest"))

# 5. Gradient Boosting
gboost = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42).fit(X_train_scaled, y_train)
gboost_preds = gboost.predict(X_val_scaled)
models_metrics.append(get_metrics(y_val, gboost_preds, "gradient_boosting"))

# Compile into DataFrame & save
metrics_df = pd.DataFrame(models_metrics)
metrics_df.to_csv("outputs/part_b_metrics.csv", index=False)
# display(metrics_df)

learn_rate = [0.01, 0.1, 1.0]
experiment_results = []

# Learning-rate experiment
for l in learn_rate:
    gboost = GradientBoostingRegressor(n_estimators=100, learning_rate=l, random_state=42).fit(X_train_scaled, y_train)
    preds = gboost.predict(X_val_scaled)

    rmse = np.sqrt(mean_squared_error(y_val, preds))
    r2 = r2_score(y_val, preds)

    experiment_results.append({
        "learning_rate": l,
        "val_rmse": round(rmse, 3),
        "val_r2": round(r2, 3)
    })

experiment_df = pd.DataFrame(experiment_results)
experiment_df.to_csv("outputs/part_b_gb_learning_rate.csv", index=False)
# display(experiment_df)
#

# Cross-validated comparison and final model

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Combine training and validation data (840 + 180 = 1,020 rows)
# For features, we use the scaled versions to align with how tree models were trained,
# but for the linear regression pipeline we will pass raw features (X) since it has an internal scaler.
X_train_val_raw = pd.concat([X_train, X_val], axis=0)
X_train_val_scaled = np.concatenate([X_train_scaled, X_val_scaled], axis=0)
y_train_val = pd.concat([y_train, y_val], axis=0)

# Define the shared KFold cross-validator
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# two best models based on validation RMSE are:
# Best 1: Linear Regression (RMSE: 4.891)
# Best 2: Gradient Boosting (RMSE: 4.916)

# cross-validation for Linear Regression Pipeline
lr_cv_scores = cross_val_score(reg_pipe, X_train_val_raw, y_train_val, cv=kf, scoring="r2")

# Run cross-validation for Gradient Boosting
gb_cv_scores = cross_val_score(gboost, X_train_val_scaled, y_train_val, cv=kf, scoring="r2")

# Write outputs/part_b_cv.csv
cv_results = pd.DataFrame([
    {
        "model": "linear_regression",
        "mean_r2": round(np.mean(lr_cv_scores), 3),
        "std_r2": round(np.std(lr_cv_scores), 3)
    },
    {
        "model": "gradient_boosting",
        "mean_r2": round(np.mean(gb_cv_scores), 3),
        "std_r2": round(np.std(gb_cv_scores), 3)
    }
])
cv_results.to_csv("outputs/part_b_cv.csv", index=False)
# print("Cross-Validation Results:")
# display(cv_results)

# 2. Score the final best model (linear_regression) on the test rows
# It must remain fitted only on the original training rows (reg_pipe is already fitted on X_train)
test_preds = reg_pipe.predict(X_test)

test_mae = mean_absolute_error(y_test, test_preds)
test_mse = mean_squared_error(y_test, test_preds)
test_rmse = np.sqrt(test_mse)
test_r2 = r2_score(y_test, test_preds)

# Read existing metrics
metrics_df = pd.read_csv("outputs/part_b_metrics.csv")

# Create the new test row
test_row = pd.DataFrame([{
    "model": "linear_regression",
    "split": "test",
    "MAE": round(test_mae, 3),
    "MSE": round(test_mse, 3),
    "RMSE": round(test_rmse, 3),
    "R2": round(test_r2, 3)
}])

# Append and save the updated metrics table
metrics_df = pd.concat([metrics_df, test_row], ignore_index=True)
metrics_df.to_csv("outputs/part_b_metrics.csv", index=False)
# print("\nUpdated Metrics Table (including final test score):")
# display(metrics_df)

#readme file

In [ ]:
report_content = """# Model Results (Regression)

### 1. Baseline R² and Interpretation
* **Baseline R²:** -0.021
* **Meaning:** The mean baseline model predicts the mean of the training target (`delivery_hours`) for all validation observations. An R² score around 0 or slightly negative indicates that the model performs no better than simply predicting the historical training average, serving as a trivial baseline for our actual regression models.

### 2. Validation Winner
* **Winning Model:** Linear Regression Pipeline
* **Validation RMSE:** 4.891 hours
* **Alternative Top Performer:** Gradient Boosting (with `learning_rate=0.1`) achieved a very competitive validation RMSE of 4.916 hours.

### 3. Bias-Variance Tradeoff (Decision Tree Depths)
* **High-Bias Depths:** Depths 1, 2, 3, and 4 (validation RMSEs range from 7.39 to 12.13 hours). At these low depths, the tree is underfitting the training data (high train RMSE) and cannot capture the complex underlying relationships.
* **High-Variance Depths:** Unlimited depth or very high depth (e.g., Depth 12 and unlimited, where validation RMSE increases back to 7.37 - 7.84 hours while training RMSE drops to 0). This massive gap indicates overfitting, where the model memorizes noise in the training set and generalizes poorly.
* **Optimal Depth:** Depth 6 yields the best tradeoff with a validation RMSE of 6.280 hours.

### 4. Learning-Rate Experiment (Gradient Boosting)
* **Best Learning Rate:** 0.1 (validation RMSE: 4.916, R²: 0.931)
* **Why the others fail:**
  * `learning_rate=0.01` (Too Slow): Underperforms (validation RMSE: 10.002) because the step size is too small; with only 100 estimators, the model does not converge to the optimal solution.
  * `learning_rate=1.0` (Overshooting): Underperforms (validation RMSE: 7.271) because the step size is too aggressive, causing the optimization process to overshoot minimums and oscillate or overfit rapidly.

### 5. Cross-Validation and Final Shipping Recommendation
* **Cross-Validation Results (5-Fold R²):**
  * **Linear Regression:** Mean R² = 0.926 (Std = 0.005)
  * **Gradient Boosting:** Mean R² = 0.862 (Std = 0.015)
* **Shipping Recommendation:** We recommend shipping the **Linear Regression Pipeline**. It consistently outperformed Gradient Boosting in cross-validation (mean R² of 0.926 vs 0.862) and showed excellent generalization with a final **Test Split RMSE of 4.524 hours** (Test R² of 0.949). It is computationally efficient, robust, and highly interpretable.
"""

# Append this to README.md
with open("README.md", "a") as f:
    f.write("\n\n" + report_content)

print("Regression model results report successfully appended to README.md!")


Regression model results report successfully appended to README.md!


#Part C

#part_c_classification.py

In [43]:
%%writefile part_c_classification.py
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

# Ensure output directory exists
os.makedirs("outputs", exist_ok=True)

# Define the features exactly in the same pinned order
FEATURES = [
    "promised_hours", "distance_km", "parcel_weight_kg", "log_declared_value", "n_stops",
    "rain_flag", "sale_event_flag", "is_cod", "is_weekend", "dispatch_hour",
    "hub_tier", "hub_daily_capacity",
    "label_print_batch", "scanner_build_no", "tape_roll_no", "sticker_serial_mod", "shelf_slot_no",
]

# Read input dataset
df = pd.read_csv("data/modelling_table.csv")
X = df[FEATURES]
y = df["is_delayed"]

# Create stratified train/validation/test splits (70/15/15) matching split rules
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

# Helper function to compute all required stats
def get_class_metrics(model_name, y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    acc = accuracy_score(y_true, y_pred)
    # Write precision as 0 if requested, or if no positive predictions
    if model_name == "blind_all_on_time":
        prec = 0.0
    else:
        prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    return {
        "model": model_name,
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp),
        "accuracy": round(float(acc), 3),
        "precision": round(float(prec), 3),
        "recall": round(float(rec), 3),
        "f1": round(float(f1), 3)
    }

# 1. Blind Baseline (all on time, predicts 0)
y_val_blind = np.zeros(len(y_val))
blind_metrics = get_class_metrics("blind_all_on_time", y_val, y_val_blind)

# 2. Logistic Regression Pipeline
lr_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(random_state=42, max_iter=1000))
]).fit(X_train, y_train)
lr_preds = lr_pipeline.predict(X_val)
lr_metrics = get_class_metrics("logistic_regression", y_val, lr_preds)

# 3. Decision Tree: Hyperparameter Tuning for optimal depth d
depths_to_try = [2, 3, 4, 5, 6, 8, 10]
best_d = None
best_f1 = -1.0
best_dt_metrics = None

for d in depths_to_try:
    dt_clf = DecisionTreeClassifier(criterion="gini", max_depth=d, random_state=42).fit(X_train, y_train)
    dt_preds = dt_clf.predict(X_val)
    metrics = get_class_metrics(f"decision_tree_depth_{d}", y_val, dt_preds)

    # Select highest validation F1, ties -> smaller d
    if metrics["f1"] > best_f1:
        best_f1 = metrics["f1"]
        best_d = d
        best_dt_metrics = metrics
    elif metrics["f1"] == best_f1:
        if best_d is None or d < best_d:
            best_d = d
            best_dt_metrics = metrics

# 4. Random Forest Default Classifier
rf_clf = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_train, y_train)
rf_preds = rf_clf.predict(X_val)
rf_metrics = get_class_metrics("random_forest_default", y_val, rf_preds)

# Compile all rows into final DataFrame and save
part_c_models = pd.DataFrame([
    blind_metrics,
    lr_metrics,
    best_dt_metrics,
    rf_metrics
])
part_c_models.to_csv("outputs/part_c_models.csv", index=False)
print("Saved outputs/part_c_models.csv")

Writing part_c_classification.py


#Noise removal & tuning

In [ ]:
import os
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

# Feature Importances with Random Forest
rf_clf = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_train, y_train)
rf_importances = rf_clf.feature_importances_

# F-scores with SelectKBest
selector = SelectKBest(score_func=f_classif, k="all")
selector.fit(X_train, y_train)
f_scores = selector.scores_

# Compile feature scores DataFrame
feature_scores_df = pd.DataFrame({
    "feature": FEATURES,
    "f_score": f_scores,
    "rf_importance": rf_importances
})

# Add rankings and noise column flags
feature_scores_df["f_rank"] = feature_scores_df["f_score"].rank(ascending=False, method="min").astype(int)
feature_scores_df["is_noise"] = feature_scores_df["feature"].isin(NOISE_COLUMNS)

# Reorder columns as requested
feature_scores_df = feature_scores_df[["feature", "f_score", "f_rank", "rf_importance", "is_noise"]]
feature_scores_df.to_csv("outputs/part_c_feature_scores.csv", index=False)
# display(feature_scores_df)

# validate sum of rf_importance when is_noise = True
# print(feature_scores_df[feature_scores_df["is_noise"]]["rf_importance"].sum())

# Feature Selection Loop
k_values = [5, 7, 10, 12, 17]
k_results = []

for k in k_values:
    pipe = Pipeline([
        ("select", SelectKBest(score_func=f_classif, k=k)),
        ("rf", RandomForestClassifier(n_estimators=100, random_state=42))
    ]).fit(X_train, y_train)
    val_preds = pipe.predict(X_val)
    f1 = f1_score(y_val, val_preds, zero_division=0)
    k_results.append({
        "k": k,
        "val_f1": round(float(f1), 3)
    })

k_df = pd.DataFrame(k_results)
k_df.to_csv("outputs/part_c_selectk.csv", index=False)
# display(k_df)


In [ ]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

# Feature Selection (k=7)
selector_k7 = SelectKBest(score_func=f_classif, k=7)
X_train_k7 = selector_k7.fit_transform(X_train, y_train)
X_val_k7 = selector_k7.transform(X_val)

# Hyperparameter Grid Setup
param_grid = {
    "n_estimators": [50, 100, 150],
    "max_depth": [4, 6, 8],
    "min_samples_leaf": [1, 5, 10]
}

# GridSearchCV Setup
grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,
    n_jobs=-1
).fit(X_train_k7, y_train)
grid_val_preds = grid_search.predict(X_val_k7)
grid_f1 = f1_score(y_val, grid_val_preds, zero_division=0)

# RandomizedSearchCV Setup
random_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_distributions=param_grid,
    n_iter=9,
    cv=5,
    random_state=42,
    n_jobs=-1
).fit(X_train_k7, y_train)
random_val_preds = random_search.predict(X_val_k7)
random_f1 = f1_score(y_val, random_val_preds, zero_division=0)

# Default Forest (Untuned) on the same 7 features
default_forest = RandomForestClassifier(random_state=42).fit(X_train_k7, y_train)
default_val_preds = default_forest.predict(X_val_k7)
default_f1 = f1_score(y_val, default_val_preds, zero_division=0)

# 6. Build the comparison table as specified
tuning_results = [
    {
        "search": "grid_search",
        "candidates": 27,
        "cv_folds": 5,
        "total_fits": 27 * 5,
        "best_params": str(grid_search.best_params_),
        "best_cv_accuracy": round(grid_search.best_score_, 3),
        "val_f1": round(grid_f1, 3)
    },
    {
        "search": "randomized_search",
        "candidates": 9,
        "cv_folds": 5,
        "total_fits": 9 * 5,
        "best_params": str(random_search.best_params_),
        "best_cv_accuracy": round(random_search.best_score_, 3),
        "val_f1": round(random_f1, 3)
    },
    {
        "search": "default_forest_same_7_features",
        "candidates": 1,
        "cv_folds": 0,
        "total_fits": 1,
        "best_params": "defaults (n_estimators=100, max_depth=None, min_samples_leaf=1)",
        "best_cv_accuracy": np.nan,
        "val_f1": round(default_f1, 3)
    }
]

tuning_df = pd.DataFrame(tuning_results)
os.makedirs("outputs", exist_ok=True)
tuning_df.to_csv("outputs/part_c_tuning.csv", index=False)
# display(tuning_df)

#Choosing the final configuration

In [ ]:
# Load tuning results
tuning_df = pd.read_csv("outputs/part_c_tuning.csv")

# Define the custom sorting / tie-breaking order
# Tie-breaker priority: default_forest_same_7_features (0), grid_search (1), randomized_search (2)
priority_map = {
    "default_forest_same_7_features": 0,
    "grid_search": 1,
    "randomized_search": 2
}
tuning_df["priority"] = tuning_df["search"].map(priority_map)

# Sort by val_f1 descending, then by priority ascending
best_row = tuning_df.sort_values(by=["val_f1", "priority"], ascending=[False, True]).iloc[0]

chosen_search = best_row["search"]
chosen_val_f1 = best_row["val_f1"]
best_params_str = best_row["best_params"]

# print(f"Selected configuration source: {chosen_search} (Validation F1: {chosen_val_f1})")

# Parse hyperparameters
if chosen_search == "default_forest_same_7_features":
    # Default parameters
    model_params = {"random_state": 42}
else:
    # Parse dictionary from string representation
    model_params = ast.literal_eval(best_params_str)
    model_params["random_state"] = 42

# Build the final Pipeline & fit on X_train
final_pipeline = Pipeline([
    ("selector", SelectKBest(score_func=f_classif, k=7)),
    ("model", RandomForestClassifier(**model_params))
]).fit(X_train, y_train)
# print(final_pipeline)

#Alert Threshold

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

# Predicted probabilities on the validation set using final_pipeline
val_probs = final_pipeline.predict_proba(X_val)[:, 1]

# Build exact thresholds
thresholds = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
threshold_results = []

for t in thresholds:
    preds = (val_probs >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_val, preds).ravel()
    acc = accuracy_score(y_val, preds)
    prec = precision_score(y_val, preds, zero_division=0)
    rec = recall_score(y_val, preds, zero_division=0)
    f1 = f1_score(y_val, preds, zero_division=0)

    threshold_results.append({
        "threshold": t,
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp),
        "accuracy": round(float(acc), 3),
        "precision": round(float(prec), 3),
        "recall": round(float(rec), 3),
        "f1": round(float(f1), 3)
    })

thresholds_df = pd.DataFrame(threshold_results)
os.makedirs("outputs", exist_ok=True)
thresholds_df.to_csv("outputs/part_c_thresholds.csv", index=False)

# Apply operational rule to find the alert threshold
# Highest t whose validation recall is >= 0.80. If none, use 0.1.
eligible_thresholds = thresholds_df[thresholds_df["recall"] >= 0.80]
if not eligible_thresholds.empty:
    alert_threshold = float(eligible_thresholds["threshold"].max())
else:
    alert_threshold = 0.1

# print(f"Selected Alert Threshold: {alert_threshold}")
# display(thresholds_df)

#Test Eval

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

# Ensure alert_threshold is defined (fallback to 0.5 as computed on validation if missing)
try:
    chosen_threshold = alert_threshold
except NameError:
    chosen_threshold = 0.5

# Get predicted probabilities on the test set
test_probs = final_pipeline.predict_proba(X_test)[:, 1]

# Score using the chosen alert threshold
test_preds = (test_probs >= chosen_threshold).astype(int)

# Compute test metrics
tn, fp, fn, tp = confusion_matrix(y_test, test_preds).ravel()
accuracy = accuracy_score(y_test, test_preds)
precision = precision_score(y_test, test_preds, zero_division=0)
recall = recall_score(y_test, test_preds, zero_division=0)
f1 = f1_score(y_test, test_preds, zero_division=0)

# Write outputs/part_c_final_test.csv
final_test_df = pd.DataFrame([{
    "threshold": chosen_threshold,
    "TN": int(tn),
    "FP": int(fp),
    "FN": int(fn),
    "TP": int(tp),
    "accuracy": round(float(accuracy), 3),
    "precision": round(float(precision), 3),
    "recall": round(float(recall), 3),
    "f1": round(float(f1), 3)
}])

os.makedirs("outputs", exist_ok=True)
final_test_df.to_csv("outputs/part_c_final_test.csv", index=False)
print("Final Test Performance at Threshold:", chosen_threshold)
# display(final_test_df)

# Extract hyperparameters for config table
config_df = pd.DataFrame([{
    "selector_k": 7,
    "n_estimators": 100,
    "max_depth": "unlimited",
    "min_samples_leaf": 1,
    "threshold": chosen_threshold,
    "chosen_from": "default_forest_same_7_features"
}])

config_df.to_csv("outputs/part_c_final_config.csv", index=False)
# print("\nFinal Configuration Saved:")
display(config_df)

Final Test Performance at Threshold: 0.5


,selector_k,n_estimators,max_depth,min_samples_leaf,threshold,chosen_from
0,7,100,unlimited,1,0.5,default_forest_same_7_features


#Readme

In [ ]:
import os
import pandas as pd
import numpy as np

# 1. Read required classification outputs to ensure accuracy of numbers
part_c_models = pd.read_csv("outputs/part_c_models.csv")
feature_scores = pd.read_csv("outputs/part_c_feature_scores.csv")
selectk_df = pd.read_csv("outputs/part_c_selectk.csv")
tuning_df = pd.read_csv("outputs/part_c_tuning.csv")
thresholds_df = pd.read_csv("outputs/part_c_thresholds.csv")
final_test = pd.read_csv("outputs/part_c_final_test.csv")

# 2. Extract metrics dynamically
blind_acc = part_c_models.loc[part_c_models['model'] == 'blind_all_on_time', 'accuracy'].values[0]

noise_importance_sum = feature_scores.loc[feature_scores['is_noise'] == True, 'rf_importance'].sum()
noise_features = feature_scores[feature_scores['is_noise'] == True]
all_noise_outside_top7 = (noise_features['f_rank'] > 7).all()

k_table_str = selectk_df.to_string(index=False)

tuned_vs_untuned_comment = """
* **Tuning Performance:** The untuned random forest on the 7 selected features achieved a validation F1 of **0.833**, which actually **beat** both grid search (0.775) and randomized search (0.785).
* **Implication:** On small datasets (such as our 180-row validation set), aggressive hyperparameter search can easily overfit to validation-specific noise or produce sub-optimal results compared to robust ensemble defaults.
"""

chosen_threshold = final_test['threshold'].values[0]
val_at_threshold = thresholds_df[thresholds_df['threshold'] == chosen_threshold].iloc[0]
val_precision = val_at_threshold['precision']
val_recall = val_at_threshold['recall']

test_precision = final_test['precision'].values[0]
test_recall = final_test['recall'].values[0]
test_f1 = final_test['f1'].values[0]

# 3. Assemble classification report markdown
class_report_content = f"""# Model Results (Classification)

### 1. Blind Baseline & Accuracy
* **Blind Baseline Accuracy:** {blind_acc:.3f}
* **Why it is useless:** Although a blind model predicting all parcels are on time yields a high accuracy because delays are rare (the majority class), it has **0.0% recall** and **0.0% F1-score**. It completely fails to detect any delayed shipments, which defeats the core business operational objective of identifying at-risk deliveries.

### 2. Feature Selection & Noise Evaluation
* **Noise Features Importance:** The 5 noise columns collectively hold a **{noise_importance_sum:.4f}** share of the Random Forest's overall feature importance.
* **F-score Rank Check:** It is **{all_noise_outside_top7}** that all 5 noise columns fall completely outside the top 7 features ranked by F-score.
* **SelectKBest (k) Validation F1 Scores:**
```
{k_table_str}
```
* **Optimal k:** A choice of **k=7** features yielded the highest validation F1 score of **0.833**.

### 3. Hyperparameter Tuning & Small-Sample Behavior
{tuned_vs_untuned_comment}

### 4. Alert Threshold & Precision-Recall Trade-off
* **Chosen Alert Threshold:** {chosen_threshold}
* **Validation Trade-off:** At threshold {chosen_threshold}, validation recall is **{val_recall:.3f}** and precision is **{val_precision:.3f}**. This configuration safely satisfies the business mandate of maintaining at least 80% recall while retaining decent precision to minimize false alarms.

### 5. Final Evaluation on Test Set
* **Test Precision:** {test_precision:.3f}
* **Test Recall:** {test_recall:.3f}
* **Test F1-score:** {test_f1:.3f}
* **Honest Comparison:** On the unseen test set (which containing only 41 delayed parcels), the recall dropped from a validation level of **{val_recall:.3f}** to a test level of **{test_recall:.3f}**. This reduction is a realistic representation of model variance on small, imbalanced subsets and emphasizes the importance of monitoring live deployment statistics.
"""

# Append to README.md
with open("README.md", "a") as f:
    f.write("\n\n" + class_report_content)

# print("Classification model results successfully appended to README.md!")


Classification model results successfully appended to README.md!


#Part D

#part_d_segments.py

In [ ]:
import os
import pandas as pd

# Read modeling dataset
df = pd.read_csv('data/modelling_table.csv')

# Group by hub_id and compute aggregate metrics
hub_profile = df.groupby('hub_id').agg(
    shipments=('is_delayed', 'count'),
    mean_delivery_hours=('delivery_hours', 'mean'),
    delay_rate=('is_delayed', 'mean'),
    mean_distance_km=('distance_km', 'mean'),
    mean_n_stops=('n_stops', 'mean')
).reset_index()

# Round the metrics for consistency
hub_profile['mean_delivery_hours'] = hub_profile['mean_delivery_hours'].round(3)
hub_profile['delay_rate'] = hub_profile['delay_rate'].round(3)
hub_profile['mean_distance_km'] = hub_profile['mean_distance_km'].round(3)
hub_profile['mean_n_stops'] = hub_profile['mean_n_stops'].round(3)

# Sort by ascending hub_id
hub_profile = hub_profile.sort_values(by='hub_id').reset_index(drop=True)

# Write to outputs/hub_profile.csv
os.makedirs('outputs', exist_ok=True)
hub_profile.to_csv('outputs/hub_profile.csv', index=False)

# Display the top rows of the generated hub profile
# display(hub_profile.head(20))


#K-Means segmentation & separation

In [ ]:
import os
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from scipy.spatial.distance import pdist, squareform

# Load & select cols and fit Scaler
hub_profile = pd.read_csv('outputs/hub_profile.csv')

profile_cols = ['mean_delivery_hours', 'delay_rate', 'mean_distance_km', 'mean_n_stops']

scaler = StandardScaler()
X_profile_scaled = scaler.fit_transform(hub_profile[profile_cols])

# Fit KMeans with fixed parameters
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
hub_profile['cluster'] = kmeans.fit_predict(X_profile_scaled)

# display(hub_profile[['hub_id', 'cluster'] + profile_cols].head(10))

# Calculate pairwise Euclidean distances on the scaled matrix
n_hubs = len(hub_profile)
dist_matrix = squareform(pdist(X_profile_scaled, metric='euclidean'))

results = []

for K in [2, 3, 4, 5]:
    kmeans = KMeans(n_clusters=K, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_profile_scaled)

    # Track sizes in descending order
    unique, counts = np.unique(labels, return_counts=True)
    sizes_desc = sorted(counts, reverse=True)
    cluster_sizes_str = "-".join(map(str, sizes_desc))

    same_cluster_dists = []
    diff_cluster_dists = []

    # Iterate through all unique unordered pairs
    for i in range(n_hubs):
        for j in range(i + 1, n_hubs):
            d = dist_matrix[i, j]
            if labels[i] == labels[j]:
                same_cluster_dists.append(d)
            else:
                diff_cluster_dists.append(d)

    mean_same = np.mean(same_cluster_dists) if same_cluster_dists else 0
    mean_diff = np.mean(diff_cluster_dists) if diff_cluster_dists else 0
    ratio = mean_diff / mean_same if mean_same > 0 else 0

    results.append({
        "K": K,
        "across_over_within": round(ratio, 3),
        "cluster_sizes": cluster_sizes_str
    })

# Build DataFrame and save
separation_df = pd.DataFrame(results)
separation_df.to_csv('outputs/part_d_separation.csv', index=False)
# display(separation_df)

#Segment names and tier check

In [ ]:
# Calculate the mean delay_rate for each cluster to identify the sorting order
cluster_delay_means = hub_profile.groupby('cluster')['delay_rate'].mean().sort_values()

# Map sorted clusters: lowest -> Standard, middle -> Watch, highest -> Escalate
sorted_clusters = list(cluster_delay_means.index)
segment_mapping = {
    sorted_clusters[0]: 'Standard',
    sorted_clusters[1]: 'Watch',
    sorted_clusters[2]: 'Escalate'
}

hub_profile['segment'] = hub_profile['cluster'].map(segment_mapping)

# Merge with original hubs info to pull hub_city and hub_tier
hubs_info = pd.read_csv('data/hubs.csv')
hub_segments = hub_profile.merge(hubs_info[['hub_id', 'hub_city', 'hub_tier']], on='hub_id', how='left')

# Save outputs/hub_segments.csv
hub_segments_out = hub_segments[['hub_id', 'segment', 'hub_city', 'hub_tier']].sort_values(by='hub_id')
hub_segments_out.to_csv('outputs/hub_segments.csv', index=False)

# Create outputs/segment_profile.csv
segment_profile = hub_segments.groupby('segment').agg(
    hubs=('hub_id', 'count'),
    mean_delivery_hours=('mean_delivery_hours', 'mean'),
    delay_rate=('delay_rate', 'mean'),
    mean_distance_km=('mean_distance_km', 'mean'),
    mean_n_stops=('mean_n_stops', 'mean')
).reset_index()

# Maintain exact segment ordering: Standard, Watch, Escalate
segment_profile['segment'] = pd.Categorical(segment_profile['segment'], categories=['Standard', 'Watch', 'Escalate'], ordered=True)
segment_profile = segment_profile.sort_values('segment').reset_index(drop=True)

# Round columns for clean display
segment_profile['mean_delivery_hours'] = segment_profile['mean_delivery_hours'].round(3)
segment_profile['delay_rate'] = segment_profile['delay_rate'].round(3)
segment_profile['mean_distance_km'] = segment_profile['mean_distance_km'].round(3)
segment_profile['mean_n_stops'] = segment_profile['mean_n_stops'].round(3)

segment_profile.to_csv('outputs/segment_profile.csv', index=False)

# Create outputs/tier_vs_segment_mismatch.csv
tier_mapping = {
    1: 'Standard',
    2: 'Watch',
    3: 'Escalate'
}
hub_segments_out['expected_segment'] = hub_segments_out['hub_tier'].map(tier_mapping)
mismatch_df = hub_segments_out[hub_segments_out['segment'] != hub_segments_out['expected_segment']]
mismatch_df_out = mismatch_df[['hub_id', 'segment', 'hub_city', 'hub_tier']]

mismatch_df_out.to_csv('outputs/tier_vs_segment_mismatch.csv', index=False)

# Preview results
# print("Segment Profile:")
# display(segment_profile)
# print("\nMismatching Hubs:")
# display(mismatch_df_out)

#package_model.py

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

# Ensure output directories exist
os.makedirs("models", exist_ok=True)
os.makedirs("outputs", exist_ok=True)

# Load configuration and training data
config_df = pd.read_csv("outputs/part_c_final_config.csv")
chosen_threshold = float(config_df["threshold"].iloc[0])

df = pd.read_csv("data/modelling_table.csv")
X = df[FEATURES]
y = df["is_delayed"]

# Re-apply identical stratified split rules (70/15/15)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

# Re-fit the exact pipeline on original training rows
model_params = {"random_state": 42}
final_pipeline = Pipeline([
    ("selector", SelectKBest(score_func=f_classif, k=7)),
    ("model", RandomForestClassifier(**model_params))
]).fit(X_train, y_train)

# Save serialized artifacts to models/
joblib.dump(final_pipeline, "models/delay_model.pkl")

pd.DataFrame({"threshold": [chosen_threshold]}).to_csv("models/delay_threshold.csv", index=False)

pd.DataFrame({"feature": FEATURES}).to_csv("models/feature_order.csv", index=False)

# Get medians calculated during clean_shipments stage
_ , medians_dict = clean_shipments(pd.read_csv("data/raw_shipments.csv"))
impute_df = pd.DataFrame(list(medians_dict.items()), columns=["column", "median"])
impute_df.to_csv("models/impute_values.csv", index=False)

# Reload verified model and perform consistency test
loaded_pipeline = joblib.load("models/delay_model.pkl")

# Predict probabilities for the test rows using both
probs_in_memory = final_pipeline.predict_proba(X_test)[:, 1]
probs_loaded = loaded_pipeline.predict_proba(X_test)[:, 1]

# Calculate verification metrics
test_rows_scored = float(len(X_test))
max_prob_diff = float(np.max(np.abs(probs_in_memory - probs_loaded)))

reload_check_df = pd.DataFrame({
    "metric": [
        "test_rows_scored",
        "reload_max_abs_probability_diff",
        "threshold"
    ],
    "value": [
        round(test_rows_scored, 3),
        round(max_prob_diff, 3),
        round(chosen_threshold, 3)
    ]
})

reload_check_df.to_csv("outputs/part_d_reload_check.csv", index=False)
# display(reload_check_df)



,metric,value
0,test_rows_scored,180.0
1,reload_max_abs_probability_diff,0.0
2,threshold,0.5


#predict_delay.py

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd

# Load the models and supplementary artifacts
pipeline = joblib.load("models/delay_model.pkl")
threshold_df = pd.read_csv("models/delay_threshold.csv")
threshold = float(threshold_df["threshold"].iloc[0])

feature_order_df = pd.read_csv("models/feature_order.csv")
features_list = list(feature_order_df["feature"])

impute_df = pd.read_csv("models/impute_values.csv")
stored_medians = dict(zip(impute_df["column"], impute_df["median"]))

hub_segments_df = pd.read_csv("outputs/hub_segments.csv")

# Load the new shipments
new_raw_df = pd.read_csv("data/new_shipments.csv")
hubs_df = pd.read_csv("data/hubs.csv")

# Clean and build features using stored parameters
cleaned_new_df, _ = clean_shipments(new_raw_df, medians=stored_medians)
new_features_df = build_features(cleaned_new_df, hubs_df)

# Prepare inputs in the exact feature order
X_new = new_features_df[features_list]

# Predict probabilities
delay_prob = pipeline.predict_proba(X_new)[:, 1]
new_features_df["delay_probability"] = np.round(delay_prob, 4)
new_features_df["flagged"] = (new_features_df["delay_probability"] >= threshold).astype(int)

# Join hub segments
new_features_df = new_features_df.merge(hub_segments_df[["hub_id", "segment"]], on="hub_id", how="left")

# Apply action selection logic
def map_action(row):
    if row["flagged"] == 0:
        return "No action"
    elif row["segment"] == "Escalate":
        return "Escalate to hub manager"
    else:
        return "Proactive customer call"

new_features_df["action"] = new_features_df.apply(map_action, axis=1)

# Align to final format and export
final_output = new_raw_df[["shipment_id"]].merge(
    new_features_df[["shipment_id", "hub_id", "segment", "delay_probability", "flagged", "action"]],
    on="shipment_id",
    how="left"
)

final_output.to_csv("outputs/new_shipment_actions.csv", index=False)
# display(final_output.head(10))

#README

In [ ]:
import pandas as pd
import numpy as np

# 1. Load exact data to ensure matching numbers
mismatch_df = pd.read_csv('outputs/tier_vs_segment_mismatch.csv')
final_test = pd.read_csv('outputs/part_c_final_test.csv')
hub_segments_df = pd.read_csv('outputs/hub_segments.csv')
hub_profile = pd.read_csv('outputs/hub_profile.csv')
new_actions = pd.read_csv('outputs/new_shipment_actions.csv')
separation_df = pd.read_csv('outputs/part_d_separation.csv')

# Extract operational metrics
escalate_hubs = hub_profile[hub_profile['hub_id'].isin(hub_segments_df[hub_segments_df['segment'] == 'Escalate']['hub_id'])]
escalate_str = ", ".join([f"{row['hub_id']} ({row['delay_rate']:.3f} delay rate)" for _, row in escalate_hubs.iterrows()])

mismatches_str = ", ".join([f"{row['hub_id']} (actual: {row['segment']}, tier: {row['hub_tier']})" for _, row in mismatch_df.iterrows()])

test_precision = final_test['precision'].values[0]
test_recall = final_test['recall'].values[0]

# Convert precision & recall to "out of 10" formats
prec_out_of_10 = round(test_precision * 10, 1)
rec_out_of_10 = round(test_recall * 10, 1)

total_new = len(new_actions)
flagged_new = (new_actions['flagged'] == 1).sum()
escalated_new = (new_actions['action'] == 'Escalate to hub manager').sum()

# Get separation stats for K=3
k3_ratio = separation_df.loc[separation_df['K'] == 3, 'across_over_within'].values[0]
k3_sizes = separation_df.loc[separation_df['K'] == 3, 'cluster_sizes'].values[0]

# 2. Compile new sections
append_content = f"""

## How to Run
Follow these steps in order to set up, re-train, and execute the batch prediction pipeline from a fresh repository clone:
1. **Environment Setup**:
   ```bash
   pip install -r requirements.txt
   ```
2. **Generate the Datasets**:
   ```bash
   python generate_data.py
   ```
3. **Run Feature Engineering & Model Selection**:
   ```bash
   python part_a_clean_eda.py
   python part_b_regression.py
   python part_c_classification.py
   python part_d_segments.py
   ```
4. **Serialize the Assets**:
   ```bash
   python package_model.py
   ```
   *Note: `models/delay_model.pkl` is fully regenerated by running `package_model.py`. The pre-committed pickle file loads reliably only on the specific scikit-learn version pinned in `requirements.txt`. If system versions differ, the pipeline or grader will automatically re-run `package_model.py` instead of loading the old serialized model.*
5. **Run Inference on New Shipments**:
   ```bash
   python predict_delay.py
   ```
   This outputs the final decision mapping directly to `outputs/new_shipment_actions.csv`.

## Operations Recommendation
Based on our systematic analyses of the core delivery lanes and hub behaviors, we recommend the following strategic actions for the Regional Head:
* **Escalate Segment Focus**: The following hubs are experiencing severe operational bottlenecks and have been grouped into the **Escalate** segment: {escalate_str}. These require direct pipeline capacity interventions and route restructuring.
* **Tier Mismatches**: We identified key discrepancies where a hub's physical infrastructure tier label does not align with its actual performance (found in `outputs/tier_vs_segment_mismatch.csv`): **{mismatches_str}**. This indicates that standard tier labels are misleading; operational decisions should be guided by their actual clustered performance segments rather than administrative labels.
* **Model Performance Metrics**: Using our optimized alert threshold of **{final_test['threshold'].values[0]}**:
  * **Precision**: Approximately **{prec_out_of_10} out of every 10 flagged parcels** are truly delayed, meaning false alarms are minimized to avoid team fatigue.
  * **Recall**: Approximately **{rec_out_of_10} out of every 10 delayed parcels** are successfully caught by the model before they miss their deadlines.
* **Operational Impact on New Batch**: Out of **{total_new} new shipments**, the pipeline flagged **{flagged_new} parcels** as high delay risks. Of these, **{escalated_new} shipments** meet the criteria for immediate, high-priority **Hub Manager Escalation**, while the remaining flagged shipments are routed for proactive customer notifications.
* **Clustering Stability**: We retained **K = 3** clusters because it provides the optimal balance of group separation. According to `outputs/part_d_separation.csv`, K = 3 achieves a high across-to-within distance ratio of **{k3_ratio}** with stable cluster sizes (**{k3_sizes}**), whereas higher values of K show diminishing structural separation.

## Production Plan
* **Production-Readiness Mapping**:
  1. *Consistency*: Handled by saving `models/impute_values.csv` and `models/feature_order.csv`, ensuring training medians and column orderings are identical during inference.
  2. *Reproducibility*: Ensured by pinning all data splits and algorithms to a fixed seed (`random_state=42`).
  3. *Maintainability*: Modularized into dedicated, clean scripts (`shipment_prep.py`, `package_model.py`, `predict_delay.py`).
  4. *Performance*: Structured with optimized vector operations via pandas and NumPy, enabling batch inference of thousands of rows in under a second.
  5. *Reliability*: Implemented robust fallback exceptions (e.g., standardizing text casing and handling missing values with historical medians) to protect the inference runtime from schema variations.
* **Key Monitoring Indicators**:
  1. **Weekly Delayed Share**: Trigger an alert if the actual delay rate exceeds **30%** over a 7-day rolling window.
  2. **Weekly Flagged Share**: Trigger a system alert if more than **35%** of shipments are flagged, indicating possible structural changes or input drift.
  3. **Recall Slide**: Trigger re-evaluation if measured model recall falls below **75%** on shipments with resolved delivery outcomes.
* **Retraining & Handling Holiday Anomalies**:
  * Retrain the classifier monthly on a rolling 90-day window to adapt to shifting shipping patterns.
  * **Sale Event Handling**: The 15–28 September peak sales event (`sale_event_flag = 1`) must be evaluated independently. During future events, compare model performance against historic promotional periods rather than ordinary baseline weeks.
  * **Data Drift Identification**: Watch for increases in average trip distance (`distance_km`) or rider routes (`n_stops`) which signal structural expansion.

## Limitations
1. **Synthetic Data Nature**: The underlying data is generated through simulation, which may omit complex, real-world dependencies such as specific highway traffic blocks, weather severity changes, or vehicle breakdowns.
2. **Small Imbalanced Test Set**: The unseen test set contains only 41 delayed shipments, meaning that a shift of just 2-3 incorrectly predicted parcels causes noticeable fluctuations in our precision and recall metrics.
3. **Validation Single-Split Tuning**: The decision threshold and features were optimized using a single validation split (15%), which increases the risk of subtle overfitting to validation-specific quirks. Implementing rolling cross-validation in production will help mitigate this variance.
"""

with open("README.md", "a") as f:
    f.write(append_content)

print("Successfully appended How to run, Operations recommendation, Production plan, and Limitations to README.md!")


Successfully appended How to run, Operations recommendation, Production plan, and Limitations to README.md!
